# GPU01 — خ۷ (F07) گروه ۷-الف: شبکه‌های عصبی جدولی روی L1

> بند 7.16 `doc/WBS-phase7-modeling.md` · اسپرینت C، ردیف «خ۷ شبکه‌ی عصبی» در
> `doc/progress/07-مدل‌سازی-و-تنظیم.md`.

**سه معماری، یک سؤال:** آیا شبکه‌ی عصبی روی ۷٬۵۷۹ رکورد چیزی به `lightgbm_quantile`
(قهرمان فعلی، یافته‌ی ۱۵) اضافه می‌کند؟

| مدل | چه چیزی را می‌آزماید |
|---|---|
| `mlp_quantile` | خط پایه‌ی عصبی روی همان ماتریس یک‌هات خانواده‌های دیگر |
| `mlp_entity_embedding` | بازنمایی آموخته‌شده به‌جای یک‌هات (عضو ۲ بند 7.16.1) |
| `ft_transformer` | توجه بین فیچرها (عضو ۴) — گران‌ترین |

⚠️ **انتظار پیش از اجرا صریحاً منفی است** (جدول بند 7.16: «L1 → 🔴 بازنده‌ی محتمل»،
سه شاهد ادبیات). این اجرا برای **رد کردن** یک فرضیه است، نه بردن — و طبق بند 7.16.4
نتیجه هرچه باشد با تعداد پارامتر و زمان آموزش گزارش می‌شود.

**بودجه‌ی هدف: ~۹۰ دقیقه.** هر مرحله سقف زمانی سخت دارد، پس نوت‌بوک همیشه به سلول
بسته‌بندی می‌رسد حتی اگر تنظیم ناتمام بماند (یافته‌ی ۱۲: بدون سقف زمانی، یک مدل ۷.۹
ساعت گرفت).

## سلول ۱ — نصب وابستگی‌ها

`torch`/`jax` روی کولب و کگل از پیش نصب‌اند و نسخه‌شان با درایور CUDA همان ماشین
هماهنگ است؛ نصب دوباره‌شان چند گیگابایت دانلود و گاهی ناسازگاری درایور می‌آورد.
پس فقط چیزهایی نصب می‌شوند که واقعاً نیستند. نسخه‌ی دقیق هرچه استفاده شد در سلول ۵
چاپ و در MLflow ثبت می‌شود (بازتولیدپذیری از راه **ثبت**، نه پین‌کردن).
فهرست کامل: `requirements-gpu.txt` داخل همین بسته.

In [ ]:
!pip install -q optuna mlflow

## سلول ۲ — بارگذاری بسته‌ی کد + داده

⚠️ **کد اصلی داخل نوت‌بوک نوشته نمی‌شود** (بند 7.8.4، قاعده‌ی «`notebooks/` = روایت،
`src/` = حقیقت»). این نوت‌بوک فقط `src/` را import و روایت می‌کند.

`gpu_bundle.zip` را با `python -m src.models.gpu_bundle` بسازید و در Drive بگذارید
(یا در کگل به‌عنوان Dataset آپلود کنید). داخلش: کل `src/`، چهار فایل
`data/processed/` که سلول ۳ رویشان assert می‌زند، و نتایج CPU خانواده‌های قبلی برای
جدول مقایسه.

In [ ]:
MODE = "colab"          # ← "colab" یا "kaggle"
BUNDLE_COLAB  = "/content/drive/MyDrive/phase7/gpu_bundle.zip"   # ← مسیر خودتان
BUNDLE_KAGGLE = "/kaggle/input/phase7-bundle/gpu_bundle.zip"

import os, sys, zipfile, pathlib

if MODE == "colab":
    from google.colab import drive
    drive.mount("/content/drive")
    bundle, workdir = BUNDLE_COLAB, pathlib.Path("/content/phase7")
else:
    bundle, workdir = BUNDLE_KAGGLE, pathlib.Path("/kaggle/working/phase7")

workdir.mkdir(parents=True, exist_ok=True)
with zipfile.ZipFile(bundle) as z:
    z.extractall(workdir)
os.chdir(workdir)
sys.path.insert(0, str(workdir))
print("محتوای بسته:", sorted(p.name for p in workdir.iterdir()))

## سلول ۳ — ⭐ دروازه‌ی انصاف A1 (بند 7.7.3)

**اگر هش‌ها نخوانند، نوت‌بوک همین‌جا می‌ایستد.** بدون این assert هیچ اثباتی وجود
ندارد که این اجرا روی همان foldها و همان snapshot دادهٔ خانواده‌های CPU انجام شده —
و هر run با `cv_folds_hash` نامنطبق از جدول مقایسه‌ی فاز ۷ حذف می‌شود. مقادیر زیر
از بخش «قفل فاز ۷» `doc/data_manifest.md` آمده‌اند.

In [ ]:
from src.models.gpu_runner import assert_fairness_gate

EXPECTED_CV_FOLDS_HASH      = "bd08d6f7c801ee0611121e404774251de07a480ac1589b12eb7c64f8044b78d4"
EXPECTED_DATA_SNAPSHOT_HASH = "68b4cb8517d292599b2f161f779758b9f3254d60302849f39d81650d0bd9fba0"   # data/processed/features_A_v1.parquet

from src.models.gpu_runner import load_l1
data = load_l1()

assert_fairness_gate(data, EXPECTED_CV_FOLDS_HASH, EXPECTED_DATA_SNAPSHOT_HASH)
print(data.summary())

## سلول ۴ — بذر تصادفی سراسری

`set_global_seed()` تنها منبع بذر پروژه است (`AGENTS.md`). قطعیت کامل روی GPU
تضمین‌شدنی نیست — به‌همین‌دلیل قاعده‌ی **سه seed** (A7، بند 7.16.3) در مرحله‌ی
قهرمان اجرا می‌شود و پراکندگی بین seedها خودش گزارش می‌گردد، نه پنهان.

In [ ]:
from src.config import set_global_seed
from src.models.gpu_runner import setup_torch_determinism

set_global_seed()
setup_torch_determinism(strict=False)   # strict=True بعضی op های cuDNN را می‌شکند

## سلول ۵ — سخت‌افزار

زمان‌های اجرا فقط با دانستن سخت‌افزار قابل تفسیرند (بند 7.8.2).

In [ ]:
!nvidia-smi

from src.models.gpu_runner import device_report

DEVICE = device_report()
DEVICE

## سلول ۶ — ردیابی MLflow جدا

`mlruns_gpu/` جداست تا ادغام با `mlruns/` محلی (بند 7.8.3 گام ۵) امن و قابل بازگشت
باشد. tag اجباری `compute` هم همین‌جا ست می‌شود.

In [ ]:
from src.models.gpu_runner import use_gpu_tracking

COMPUTE = "colab"     # اگر روی کگل اجرا می‌کنید: "kaggle"
print("MLflow →", use_gpu_tracking("mlruns_gpu"))

## سلول ۷-الف — R0: آزمایش دود

بند 7.3.2: یک برازش با هایپرپارامتر پیش‌فرض، فقط برای اثبات اجراپذیری — **و پیدا
کردن باگ**. `smoke_test` هر ۱۳ معیار عملیاتی را حساب می‌کند، نه فقط زمان اجرا؛
نسخه‌ی اول هارنس S0 خ۱ فقط زمان را ثبت می‌کرد و دو باگ واقعی را پنهان کرد
(یافته‌های ۰ و ۴). سیم‌چین نشتی بند 7.9.2 هم این‌جا فعال است: $R^2>0.9$ ⇒ توقف.

اگر این سلول خطا داد، **ادامه ندهید** — بقیه‌ی نوت‌بوک ۹۰ دقیقه محاسبه‌ی بی‌فایده است.

In [ ]:
from src.models.families import f07_neural as fam
from src.models.gpu_runner import smoke_test

MODEL_IDS = ["mlp_quantile", "mlp_entity_embedding", "ft_transformer"]
smoke = [smoke_test(fam.FITTERS[m], data, hyperparams={"epochs": 30, "patience": 6})
         for m in MODEL_IDS]

## سلول ۷-ب — R2: تنظیم با بودجه‌ی زمانی

Optuna TPE روی هر ۵ fold رسمی، τ=۰.۲۰ (نقطه‌ی عملیاتی پروژه، ردیف ۳۴ decision_log).
بودجه بر حسب **دقیقه** است نه تعداد trial — محدودیت واقعی کولب زمان است.

مجموع ~۶۰ دقیقه. اگر GPU سریع‌تر بود، عددها را بالا ببرید؛ اگر session ناپایدار
است پایین. مطالعه روی SQLite ذخیره می‌شود، پس اجرای دوباره **ادامه** می‌دهد و از
صفر شروع نمی‌کند (بند 7.6.3).

In [ ]:
from src.models.gpu_runner import run_gpu_study
from src.models.spaces import SPACES

BUDGET_MINUTES = {"mlp_quantile": 18, "mlp_entity_embedding": 20, "ft_transformer": 24}

studies = []
for mid in MODEL_IDS:
    studies.append(run_gpu_study(
        fam.FITTERS[mid], SPACES[mid].fn, data,
        family=fam.FAMILY, feature_set=fam.FEATURE_SET,
        budget_minutes=BUDGET_MINUTES[mid], compute=COMPUTE, seed=42))

## سلول ۷-ج — قهرمان: سه seed + کالیبراسیون ACI + آزمون Diebold-Mariano

فقط بهترین مدل قهرمان می‌شود (بودجه محدود است). سه چیز این‌جا اتفاق می‌افتد که
بدون آن‌ها «برد» یک ادعای بی‌پشتوانه است:

1. **سه seed** (قاعده‌ی A7) — قطعیت GPU تضمین‌شدنی نیست؛ اگر دامنه‌ی بین seedها از
   فاصله تا B3 بزرگ‌تر باشد، «برد» نویز است.
2. **ACI** (بند 7.22.1 عضو ۴) — یافته‌ی ۲۲: تنها لایه‌ی کالیبراسیونی که روی این داده
   کار کرد (CQR ایستا هر ۴ قهرمان را بدتر کرد، یافته‌ی ۲۰).
3. **DM-test** در برابر B3 روی زیان **تک‌ردیفی** (بند ۶.۶) — یافته‌ی ۱۳: بدون این،
   ادعای «سه مدل خ۱ B3 را بردند» تأیید نشد.

مدل هر fold/seed در `models/gpu/F07/` ذخیره می‌شود و بعد از این سلول بازخوانی و
راستی‌آزمایی می‌گردد.

In [ ]:
import numpy as np
from src.models.gpu_runner import finalize_champion

best = min(studies, key=lambda s: s.best_pinball)
print(f"قهرمان: {best.model_id} (pinball={best.best_pinball:.5f})\n")

champions = [finalize_champion(fam.FITTERS[best.model_id], data, best,
                               feature_set=fam.FEATURE_SET, seeds=(42, 1234, 2026),
                               compute=COMPUTE, run_aci=True)]

## سلول ۷-د — راستی‌آزمایی مدل ذخیره‌شده

مدل سنگین فقط وقتی «ذخیره‌شده» حساب می‌شود که **بازخوانی‌اش همان عدد را بدهد**.
این سلول یکی از فایل‌های ذخیره‌شده را از دیسک بارمی‌گرداند و پیش‌بینی‌اش را با
پیش‌بینی مدل تازه‌برازش‌شده مقایسه می‌کند. اگر این تست پاس نشود، فایل‌های داخل zip
بی‌ارزش‌اند.

In [ ]:
from pathlib import Path
from src.models.axes import TUNING_TAU

stem = Path(f"models/gpu/F07/{best.model_id}/{best.model_id}__s42__fold0")
reloaded = fam.FITTERS[best.model_id].load(stem)
_, test0 = data.folds[0]
pred_reloaded = reloaded.predict(test0, TUNING_TAU)

print(f"پارامترها: {reloaded.n_parameters:,}")
print(f"پیش‌بینی از مدل بازخوانی‌شده — میانگین={pred_reloaded.mean():.5f} "
      f"· min={pred_reloaded.min():.5f} · max={pred_reloaded.max():.5f}")
print(f"فایل‌های ذخیره‌شده: {len(list(stem.parent.glob('*')))}")
assert np.isfinite(pred_reloaded).all(), "مدل بازخوانی‌شده خروجی نامعتبر داد"
print("✅ مدل ذخیره‌شده قابل استفاده است")

## سلول ۷-ه — جدول اجباری بند 7.16.4: پیچیدگی در برابر بهره

⭐ **ستون «Δ نسبت به LightGBM» عمدی است.** اگر شبکه‌ها نبرند، گزارش باید همین را
به‌عنوان یافته بنویسد، نه پنهانش کند. عدد LightGBM از `reports/phase7/S2_tuning_F02.json`
همین بسته خوانده می‌شود (نه کپی دستی که کهنه می‌شود).

In [ ]:
import json, pandas as pd
from pathlib import Path

lgbm = json.loads(Path("reports/phase7/S2_tuning_F02.json").read_text())
lgbm_pinball = lgbm["lightgbm_quantile"]["best_pinball"]

rows = []
for st, sm in zip(studies, smoke):
    rows.append({
        "مدل": st.model_id,
        "بهترین pinball": round(st.best_pinball, 5),
        "B3": round(st.baseline_b3, 5),
        "LightGBM (خ۲)": round(lgbm_pinball, 5),
        "Δ نسبت به LightGBM": round(st.best_pinball - lgbm_pinball, 5),
        "trial": st.n_trials_done,
        "ساعت-هسته": round(st.seconds / 3600, 2),
        "زمان یک برازش (R0)": round(sm["seconds"], 1),
    })
complexity_table = pd.DataFrame(rows).sort_values("بهترین pinball")
complexity_table

## سلول ۷-و — گزارش فارسی کامل

همه‌چیز روی دیسک نوشته می‌شود چون session کولب از بین می‌رود. این فایل مستقیماً
مبنای کارت مدل ۱۴ گامی و ردیف `doc/progress/07-*.md` است.

In [ ]:
from src.models.gpu_runner import render_family_report, save_family_report

notes = [
    f"جدول ۷.۱۶.۴ (پیچیدگی در برابر بهره) در همین گزارش: بهترین شبکه "
    f"{min(s.best_pinball for s in studies):.5f} در برابر LightGBM {lgbm_pinball:.5f}.",
    "انتظار پیش از اجرا (بند 7.16) منفی بود؛ نتیجه هرچه باشد باید صریح گزارش شود.",
    "سر چند-کوانتایلی با cumsum نامنفی ⇒ تقاطع کوانتایل ساختاراً ناممکن (بند 7.23.1).",
    "دسته‌بندی بلوکی روزانه اعمال شد (ICC روز=۰.۲۲۵، F10) — نه دسته‌ی تصادفی ردیفی.",
]
report = render_family_report(
    "F07", "خ۷ گروه ۷-الف — شبکه‌های عصبی جدولی روی L1 (اجرای GPU)",
    studies, champions, smoke, DEVICE, notes)
save_family_report("F07", report, "F07a_neural_L1")
complexity_table.to_csv("reports/gpu/F07a_complexity_vs_gain.csv", index=False)
print(report)

## سلول ۸ — بسته‌بندی خروجی (تکه‌های ۱۰۰ مگابایتی)

همه‌ی خروجی‌ها — `mlruns_gpu/` (هر trial + قهرمان‌ها با artifact مدل)،
`models/gpu/` (وزن‌ها و پیش‌پردازش هر fold/seed)، `reports/gpu/` (JSON و گزارش
فارسی)، و `optuna_studies/*.db` (تا اجرای بعدی از همین‌جا ادامه دهد) — در یک zip
جمع و به تکه‌های ۱۰۰ مگابایتی شکسته می‌شوند. هر تکه SHA-256 خودش را در
`MANIFEST_F07a_neural_L1.json` دارد، پس اگر دانلود یکی خراب شد فقط همان یکی دوباره گرفته
می‌شود.

In [ ]:
from src.models.gpu_runner import package_outputs, download_parts

manifest = package_outputs(tag="F07a_neural_L1", part_mb=100)
download_parts()      # روی کولب دانلود می‌کند؛ روی کگل فایل‌ها در خروجی session می‌مانند

---
## پس از اجرا — چرخه‌ی بازگشت (بند 7.8.3)

```
۱. این نوت‌بوک اجراشده (File → Download .ipynb، با تمام خروجی‌ها) →
   notebooks/gpu/executed/{name}__{تاریخ}.ipynb    ← حتی اگر آزمایش شکست خورد؛ شکست هم داده است
۲. تکه‌ها → ریشه‌ی مخزن:  cat gpu_outputs_*.zip.part* > gpu_outputs.zip && unzip gpu_outputs.zip
۳. rsync -a mlruns_gpu/ mlruns/        (ادغام MLflow)
۴. کارت مدل ۱۴ گامی → reports/models/{model_id}.md
۵. make mlflow-ui  →  runهای جدید با tag compute=colab باید دیده شوند
```